# Hybrid Search: BM25 (Sparse) + Concept Embeddings (Dense Stand-In) with Reciprocal Rank Fusion

This notebook implements **HybridSearch RAG** from scratch (see `../03-hybrid-search-rag.md`), the
technique used for the Virtual Liaison's cost-catalog RAG, where clients mix natural language
("how much for a rush Japanese translation") with exact identifiers ("price for SKU-4471").

We implement:
1. **BM25** from scratch -- strong on exact token matches, blind to synonyms/paraphrase.
2. A small **concept-embedding** stand-in for a dense/semantic retriever -- strong on paraphrase,
   blind to arbitrary identifier tokens it has no learned meaning for (exactly how a real embedding
   model treats a rare SKU string).
3. **Reciprocal Rank Fusion (RRF)** combining both.
4. Two small synthetic queries, each designed to break *one* of the two methods individually, showing
   the **fused ranking is the only one that gets both right**.

Fully offline -- only `numpy`, `pandas`, and the standard library are used (no `rank_bm25` package
required, no API keys).

In [1]:
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
print("Ready.")

Ready.


## 1. Synthetic cost-catalog corpus

Each entry has a SKU-style identifier plus a short natural-language description. Note that the
"expedited" package never uses the word "rush", and the "localization" packages never use the word
"translation" -- on purpose, to later demonstrate BM25's blindness to synonyms.

In [2]:
catalog = [
    {"id": "SKU-LOC-JP-STD", "text": "SKU-LOC-JP-STD japanese localization standard package"},
    {"id": "SKU-LOC-JP-EXP", "text": "SKU-LOC-JP-EXP japanese localization expedited package priority handling"},
    {"id": "SKU-LOC-KR-STD", "text": "SKU-LOC-KR-STD korean localization standard package"},
    {"id": "SKU-REG-EU-STD", "text": "SKU-REG-EU-STD regulatory compliance submission standard"},
    {"id": "SKU-4471", "text": "SKU-4471 oncology campaign bundle localization compliance"},
]

catalog_df = pd.DataFrame(catalog)
catalog_df

,id,text
0,SKU-LOC-JP-STD,SKU-LOC-JP-STD japanese localization standard package
1,SKU-LOC-JP-EXP,SKU-LOC-JP-EXP japanese localization expedited package priority handling
2,SKU-LOC-KR-STD,SKU-LOC-KR-STD korean localization standard package
3,SKU-REG-EU-STD,SKU-REG-EU-STD regulatory compliance submission standard
4,SKU-4471,SKU-4471 oncology campaign bundle localization compliance


## 2. BM25 from scratch

BM25 scores a query against each document using term frequency, inverse document frequency, and
document-length normalization -- pure lexical/token overlap, no notion of "meaning" (see Chapter 3).

In [3]:
class SimpleBM25:
    """Minimal BM25 (Okapi) implementation -- no external dependency required.

    Equivalent in spirit to `rank_bm25.BM25Okapi`, reimplemented here so this notebook has zero
    dependencies beyond numpy/pandas.
    """

    def __init__(self, corpus_tokens: list[list[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus_tokens
        self.doc_lens = [len(doc) for doc in corpus_tokens]
        self.avgdl = sum(self.doc_lens) / len(corpus_tokens)
        self.doc_freqs = [Counter(doc) for doc in corpus_tokens]
        self.n_docs = len(corpus_tokens)
        self.idf = self._compute_idf()

    def _compute_idf(self) -> dict:
        df = defaultdict(int)
        for doc in self.corpus:
            for term in set(doc):
                df[term] += 1
        return {
            term: math.log(1 + (self.n_docs - freq + 0.5) / (freq + 0.5))
            for term, freq in df.items()
        }

    def get_scores(self, query_tokens: list[str]) -> list[float]:
        scores = [0.0] * self.n_docs
        for i, doc_freq in enumerate(self.doc_freqs):
            doc_len = self.doc_lens[i]
            for term in query_tokens:
                if term not in doc_freq:
                    continue
                freq = doc_freq[term]
                idf = self.idf.get(term, 0.0)
                denom = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                scores[i] += idf * (freq * (self.k1 + 1)) / denom
        return scores


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9\-]+", text.lower())


tokenized_corpus = [tokenize(doc["text"]) for doc in catalog]
bm25 = SimpleBM25(tokenized_corpus)


def bm25_search(query: str) -> list[tuple[str, float]]:
    """Only returns documents with a nonzero score -- a document BM25 has zero lexical overlap
    with is a document BM25 didn't retrieve at all, not a meaningfully-ranked last place."""
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(catalog_df["id"], scores), key=lambda t: t[1], reverse=True)
    return [(doc_id, s) for doc_id, s in ranked if s > 1e-9]


print("BM25 index built over", len(tokenized_corpus), "catalog entries.")

BM25 index built over 5 catalog entries.


## 3. A concept-embedding stand-in for dense retrieval

A real embedding model represents *meaning* -- "translation" and "localization" land close together
in vector space, but an arbitrary internal code like `SKU-4471` gets a weak, almost noise-like
representation because the model has no learned concept for it.

We simulate exactly that behavior with a small, hand-built **concept dictionary**: known
domain concepts (languages, service types, urgency) map to a fixed direction in an 8-dimensional
space, and synonyms map to the *same* concept (`translation` and `localization` both map to the
`localization` concept). Any token **not** in the dictionary -- including every SKU code -- contributes
nothing, mirroring an embedding model's weak signal on out-of-vocabulary identifiers. A document's
embedding is the (normalized) sum of its known-concept vectors; the same function embeds queries.

In [4]:
CONCEPTS = ["japanese", "korean", "regulatory", "localization", "expedited", "standard"]
CONCEPT_DIM = len(CONCEPTS)

# Synonym -> concept. Everything NOT listed here (including every SKU token) is out-of-vocabulary
# and contributes a zero vector -- exactly how a real embedding model treats an arbitrary code.
SYNONYM_TO_CONCEPT = {
    "japanese": "japanese",
    "korean": "korean",
    "regulatory": "regulatory", "compliance": "regulatory", "submission": "regulatory",
    "localization": "localization", "translation": "localization", "translate": "localization",
    "expedited": "expedited", "rush": "expedited", "urgent": "expedited", "fast": "expedited",
    "standard": "standard", "regular": "standard",
}

def concept_embed(text: str) -> np.ndarray:
    vec = np.zeros(CONCEPT_DIM)
    for token in tokenize(text):
        concept = SYNONYM_TO_CONCEPT.get(token)
        if concept is not None:
            vec[CONCEPTS.index(concept)] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


def dense_search(query: str) -> list[tuple[str, float]]:
    """Only returns documents with a nonzero concept overlap -- if the query embeds to (near) the
    zero vector (no recognized concept words), dense retrieval has no real signal and should
    contribute nothing, not an arbitrary tie-broken full ranking."""
    q_vec = concept_embed(query)
    scored = []
    for doc in catalog:
        d_vec = concept_embed(doc["text"])
        sim = float(np.dot(q_vec, d_vec))   # both already L2-normalized -> dot == cosine
        scored.append((doc["id"], sim))
    ranked = sorted(scored, key=lambda t: t[1], reverse=True)
    return [(doc_id, s) for doc_id, s in ranked if s > 1e-9]


print("Concept embedding demo:", concept_embed("rush japanese translation package"))

Concept embedding demo: [0.57735027 0.         0.         0.57735027 0.57735027 0.        ]


## 4. Reciprocal Rank Fusion

Combine two independent rankings by **rank position**, not raw score -- avoids the problem of BM25
scores and cosine similarities living on incomparable scales (see Chapter 3).

In [5]:
def reciprocal_rank_fusion(rankings: list[list[str]], k: int = 60) -> dict[str, float]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))


def hybrid_retrieve(query: str):
    dense_ranked = [doc_id for doc_id, _ in dense_search(query)]
    bm25_ranked = [doc_id for doc_id, _ in bm25_search(query)]
    fused = reciprocal_rank_fusion([dense_ranked, bm25_ranked])
    return dense_ranked, bm25_ranked, list(fused.keys())


def rank_of(doc_id: str, ranking: list[str]):
    """Returns 1-based rank, or None if the method didn't retrieve this document at all
    (i.e. zero lexical/semantic overlap -- a real 'not found', not an arbitrary last place)."""
    return ranking.index(doc_id) + 1 if doc_id in ranking else None

## 5. Query A: exact-identifier lookup (breaks dense retrieval)

`"price for SKU-4471"` contains no recognized domain concept words at all -- every token is either a
stopword-ish filler ("price", "for") or the literal SKU code. The concept embedding for this query is
therefore the **zero vector**: it has zero cosine similarity with every document, so dense retrieval
retrieves **nothing at all** for this query -- exactly how a real embedding model behaves when a
query has no content it has a learned representation for. BM25, in contrast, matches the literal
`sku-4471` token exactly and retrieves it immediately.

In [6]:
query_a = "price for SKU-4471"
dense_ranked_a, bm25_ranked_a, fused_a = hybrid_retrieve(query_a)

print("Dense-only ranking: ", dense_ranked_a if dense_ranked_a else "(nothing retrieved)")
print("BM25-only ranking:  ", bm25_ranked_a)
print("Fused ranking:      ", fused_a)
print()
r_dense = rank_of("SKU-4471", dense_ranked_a)
r_bm25 = rank_of("SKU-4471", bm25_ranked_a)
r_fused = rank_of("SKU-4471", fused_a)
print("Rank of SKU-4471 -- dense:", r_dense if r_dense else "not retrieved",
      "| BM25:", r_bm25, "| fused:", r_fused)

Dense-only ranking:  (nothing retrieved)
BM25-only ranking:   ['SKU-4471']
Fused ranking:       ['SKU-4471']

Rank of SKU-4471 -- dense: not retrieved | BM25: 1 | fused: 1


## 6. Query B: paraphrase (breaks BM25)

`"rush japanese translation package"` never uses the literal words in the `SKU-LOC-JP-EXP` catalog
entry ("expedited", not "rush"; "localization", not "translation") -- so on pure lexical overlap it
only shares "japanese" and "package" with the query, the same two terms the *standard* package also
shares. Worse, `SKU-LOC-JP-EXP`'s description is a touch longer (it also mentions "priority
handling"), so BM25's document-length normalization actually penalizes it slightly relative to the
shorter standard-package entry -- BM25 alone ranks the *wrong* package first. The concept embedding,
however, correctly maps "rush" -> expedited and "translation" -> localization regardless of document
length, so dense retrieval identifies `SKU-LOC-JP-EXP` as the clear best match.

In [7]:
query_b = "rush japanese translation package"
dense_ranked_b, bm25_ranked_b, fused_b = hybrid_retrieve(query_b)

print("Dense-only ranking: ", dense_ranked_b)
print("BM25-only ranking:  ", bm25_ranked_b)
print("Fused ranking:      ", fused_b)
print()
print("Rank of SKU-LOC-JP-EXP -- dense:", rank_of("SKU-LOC-JP-EXP", dense_ranked_b),
      "| BM25:", rank_of("SKU-LOC-JP-EXP", bm25_ranked_b),
      "| fused:", rank_of("SKU-LOC-JP-EXP", fused_b))

Dense-only ranking:  ['SKU-LOC-JP-EXP', 'SKU-LOC-JP-STD', 'SKU-4471', 'SKU-LOC-KR-STD']
BM25-only ranking:   ['SKU-LOC-JP-STD', 'SKU-LOC-JP-EXP', 'SKU-LOC-KR-STD']
Fused ranking:       ['SKU-LOC-JP-EXP', 'SKU-LOC-JP-STD', 'SKU-LOC-KR-STD', 'SKU-4471']

Rank of SKU-LOC-JP-EXP -- dense: 1 | BM25: 2 | fused: 1


## 7. Summary: hybrid is the only method that gets both queries right

Each pure method has a blind spot; the fused ranking recovers the correct top-1 answer in both cases,
because whichever method didn't have the blind spot on a given query pulls the right answer up.

In [8]:
def fmt_rank(r):
    return "not retrieved" if r is None else r


summary = pd.DataFrame([
    {
        "query": "A: exact SKU lookup", "target": "SKU-4471",
        "dense_rank": fmt_rank(rank_of("SKU-4471", dense_ranked_a)),
        "bm25_rank": fmt_rank(rank_of("SKU-4471", bm25_ranked_a)),
        "fused_rank": fmt_rank(rank_of("SKU-4471", fused_a)),
    },
    {
        "query": "B: paraphrase ('rush'/'translation')", "target": "SKU-LOC-JP-EXP",
        "dense_rank": fmt_rank(rank_of("SKU-LOC-JP-EXP", dense_ranked_b)),
        "bm25_rank": fmt_rank(rank_of("SKU-LOC-JP-EXP", bm25_ranked_b)),
        "fused_rank": fmt_rank(rank_of("SKU-LOC-JP-EXP", fused_b)),
    },
])
summary

,query,target,dense_rank,bm25_rank,fused_rank
0,A: exact SKU lookup,SKU-4471,not retrieved,1,1
1,B: paraphrase ('rush'/'translation'),SKU-LOC-JP-EXP,1,2,1


## Takeaways

- **Query A** (exact SKU): dense retrieval had **zero** usable signal -- it retrieved nothing at all
  -- while BM25 nailed the exact token match at rank 1. Because RRF only combines documents each
  method actually retrieved, the fused ranking inherits BM25's win outright (fused rank 1).
- **Query B** (paraphrase): BM25 couldn't tell the expedited package apart from the standard one
  (neither "rush" nor "translation" appear literally in either document) and, once document-length
  normalization is factored in, actually ranked the *wrong* package first. The concept/dense
  embedding correctly mapped the synonyms regardless of document length and ranked the right SKU
  first. Fusion inherits dense's win (fused rank 1).
- **Neither pure method wins both queries; the fused ranking does.** This is the concrete version of
  the claim in `../03-hybrid-search-rag.md`: hybrid search earns its added complexity specifically
  when the query mix includes both exact identifiers and natural-language paraphrase, which is exactly
  the mix a cost-catalog and project-codename system produces.